# 6단계: 추천 평가

5단계 설정(텍스트 A/B, 모드, 상한, 후보 수, 가중치)을 정답 세트로 비교한다.

정답 세트
1. 여러 설정의 상위 5개를 합쳐 질의별 판정 풀을 만든다.
2. 각 (질의, 메뉴)에 적합도를 매긴다. 2 적합, 1 부분, 0 부적합.
3. 현재 판정은 Claude가 메뉴명, 업체명, 분류만 보고 매긴 모델 추정이며 전부 검토대기다. 저장된 라벨은 보지 않았다.
4. `data/processed/evaluation/judgments.csv`에서 적합도를 고치고 검토상태를 승인으로 바꾸면 승인 판정만으로 다시 계산한다.

주의
- 후보는 공공 데이터만이다. 지표는 모델 추정 정답 기준이며 최종 성능이 아니다.
- 풀에 없는 항목은 미판정이며 0으로 계산한다. 미판정 비율을 함께 본다.
- 질의 40개라 작은 차이는 부트스트랩 신뢰구간으로 확인한다.

## 1. 모듈 로드

In [1]:
import json
import sys
import time
from collections import Counter
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src").is_dir() and (p / "data").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Menu_recommend 저장소 안에서 실행해 주세요")
sys.path.insert(0, str(PROJECT_ROOT))

from src.embedding import DEFAULT_SPEC, E5Embedder, describe_environment
from src.preprocessing import parse_query
from src.ranking import RankingConfig
from src.recommendation import EMBEDDING_ONLY, FILTER_ONLY, FULL, PipelineConfig, Recommender
from src.recommendation.evaluation import (
    APPROVED, PENDING, UNJUDGED, build_pool, evaluate_configs, evaluate_per_query, evaluate_result, judgment_map,
    load_judgments, merge_pool, paired_bootstrap, save_judgments,
)
from src.retrieval import load_index

REC_DIR = PROJECT_ROOT / "data" / "processed" / "recommendation"
OUT_DIR = PROJECT_ROOT / "data" / "processed" / "evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
JUDGMENTS_PATH = OUT_DIR / "judgments.csv"
K = 5

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)
describe_environment()

{'platform': 'macOS-26.6.2-arm64-arm-64bit',
 'processor': 'arm',
 'cpu_count': 12,
 'total_memory_gb': 32.0,
 'torch_version': '2.14.0',
 'cuda_available': False,
 'mps_available': True,
 'selected_device': 'mps'}

## 2. 평가 질의

5단계 질의 18개와 추가 질의 22개. 모순, 빈 입력, "매운 것도 괜찮아"는 정답을 정할 수 없어 제외한다.

In [2]:
run5 = json.load(open(REC_DIR / "run_config.json", encoding="utf-8"))
EXCLUDED = {"": "빈 입력, 0건 반환이 정답", "맵지 않은 매운 음식": "모순, 0건 반환이 정답",
            "국물 없는 국물 요리": "모순, 0건 반환이 정답", "매운 것도 괜찮아": "조건 없음, 적합 기준을 정할 수 없음"}
EXTRA_QUERIES = {
    "부정": ["국물 없는 담백한 거", "튀김 아닌 걸로 가볍게", "느끼한 거 말고 얼큰한 거", "뜨겁지 않은 음식", "무조건 안 매운 걸로"],
    "복합": ["따뜻하고 든든한 국밥", "차가운 면 요리", "매콤한 볶음 요리", "구운 고기 요리", "안 맵고 국물 있는 면", "꼭 국물 요리로", "찜 요리 든든하게"],
    "메뉴언급": ["치킨 먹고 싶어", "피자 말고 버거", "샌드위치 가볍게", "떡볶이 매운 거", "김밥이랑 국물"],
    "맥락": ["추운 날 뜨끈한 국밥", "야식으로 가벼운 거", "해장되는 얼큰한 국", "비 오는 날 전 부쳐 먹고 싶다", "다이어트 중이라 가벼운 샐러드"],
}
QUERY_KIND = {q["질의"]: f"5단계 {q['유형']}" for q in run5["질의"]}
QUERY_KIND.update({q: f"추가 {kind}" for kind, qs in EXTRA_QUERIES.items() for q in qs})
QUERIES = [q["질의"] for q in run5["질의"] if q["질의"] not in EXCLUDED] + [q for qs in EXTRA_QUERIES.values() for q in qs]
pd.DataFrame([{"질의": q, "유형": QUERY_KIND[q]} for q in QUERIES]).to_csv(OUT_DIR / "queries.csv", index=False, encoding="utf-8-sig")
print(f"평가 질의 {len(QUERIES)}개 (5단계 {len(QUERIES) - 22}, 추가 22), 제외 {len(EXCLUDED)}개")
pd.DataFrame([{"질의": q or "(빈 입력)", "제외 사유": r} for q, r in EXCLUDED.items()])

평가 질의 40개 (5단계 18, 추가 22), 제외 4개


,질의,제외 사유
0,(빈 입력),"빈 입력, 0건 반환이 정답"
1,맵지 않은 매운 음식,"모순, 0건 반환이 정답"
2,국물 없는 국물 요리,"모순, 0건 반환이 정답"
3,매운 것도 괜찮아,"조건 없음, 적합 기준을 정할 수 없음"


## 3. 추천기 로드 (텍스트 A, B)

In [3]:
t0 = time.time()
embedder = E5Embedder(spec=DEFAULT_SPEC)
encode_query = lambda text: embedder.encode_queries([text], show_progress=False)[0]
RECS = {}
for variant in ("A", "B"):
    index, ref = load_index(variant)
    RECS[variant] = Recommender(index, encode_query, ref)
    print(f"텍스트 {variant}: {ref['name']} 후보 {index.size}건 / 전체 {ref['후보범위']['전체수']}건 (프랜차이즈 제외)")
print(f"로드 {time.time() - t0:.1f}초, device={embedder.device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/Users/jack/project/Menu-recommend-algorithmn/src/embedding/embedder.py:123: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  loaded_dim = self.model.get_sentence_embedding_dimension()


텍스트 A: multilingual-e5-base_d1287505_textA_v1_eb332ded 후보 1088건 / 전체 5219건 (프랜차이즈 제외)
텍스트 B: multilingual-e5-base_d1287505_textB_v1_f87e895f 후보 1088건 / 전체 5219건 (프랜차이즈 제외)
로드 7.4초, device=mps


## 4. 판정 풀과 판정 시트

A와 B의 3개 모드, B의 상한 0/1, 감점 0.02, 후보 100/400 고정, 메뉴 가점 0/0.3을 풀에 넣는다.
기존 판정은 그대로 두고 새 항목만 미판정으로 덧붙인다.

In [4]:
MODES = {"임베딩만": EMBEDDING_ONLY, "조건적용": FILTER_ONLY, "조건+재랭킹+중복제어": FULL}
VARIANTS = {
    "상한0": PipelineConfig(ranking=RankingConfig(group_cap=0)),
    "상한1": PipelineConfig(ranking=RankingConfig(group_cap=1)),
    "감점0.02": PipelineConfig(ranking=RankingConfig(group_penalty=0.02)),
    "후보100 고정": PipelineConfig(candidate_k=100, preference_widen_k=100),
    "후보400 고정": PipelineConfig(candidate_k=400, preference_widen_k=400),
    "메뉴가점0": PipelineConfig(ranking=RankingConfig(menu_match_weight=0.0)),
    "메뉴가점0.3": PipelineConfig(ranking=RankingConfig(menu_match_weight=0.3)),
}
pool = build_pool([RECS["A"], RECS["B"]], QUERIES, list(MODES.values()), k=K)
seen = {(p["질의"], p["라벨링단위ID"]) for p in pool}
pool += [p for p in build_pool([RECS["B"]], QUERIES, list(VARIANTS.values()), k=K)
         if (p["질의"], p["라벨링단위ID"]) not in seen]

existing = load_judgments(JUDGMENTS_PATH)
criteria = {r["질의"]: r["판정기준"] for r in existing if r.get("판정기준")}
rows = merge_pool(existing, pool, criteria)
save_judgments(JUDGMENTS_PATH, rows)
status = Counter(r["검토상태"] for r in rows)
print(f"풀 {len(pool)}건, 시트 {len(rows)}행 (기존 {len(existing)}, 새로 추가 {len(rows) - len(existing)})")
print("검토상태:", dict(status))
print("판정출처:", dict(Counter(r["판정출처"] for r in rows if r["판정출처"])))

풀 600건, 시트 1272행 (기존 1272, 새로 추가 0)
검토상태: {'검토대기': 1272}
판정출처: {'모델추정 (Claude, 메뉴명·업체명·분류만 참조)': 1272}


In [5]:
# 질의별 판정 기준
pd.DataFrame([{"질의": q, "판정기준": criteria.get(q, "-")} for q in QUERIES])

,질의,판정기준
0,비 오는 날 얼큰한 국물 먹고 싶어,"2: 매콤한 뜨거운 국물 요리 / 1: 국물 요리지만 안 매움, 또는 매운 비국물 / 0: 그 외"
1,맵지 않고 따뜻한 음식,2: 안 맵고 따뜻하게 먹는 음식 / 1: 안 맵지만 차갑거나 상온 / 0: 매운 음식
2,차갑고 가볍게 먹을 메뉴,"2: 차갑고 가벼운 음식(냉면·냉국·샐러드·아이스류) / 1: 카페 냉장 샌드위치, 가볍지만 따뜻함, 차갑지만 무거움 / 0: 뜨겁고 무거움"
3,바삭하고 기름진 음식,2: 튀김·치킨·튀긴 패티 / 1: 기름지지만 바삭하지 않음(피자 등) / 0: 국·면·죽
4,든든한 밥 한 끼,"2: 밥 중심 한 끼(덮밥·볶음밥·비빔밥·국밥) / 1: 밥 아닌 든든한 식사, 흰밥·김밥 / 0: 간식"
5,국물 없는 매운 음식,"2: 매운 비국물 음식 / 1: 약간 매운 비국물, 안 매운 비국물 / 0: 국물 요리"
6,상큼하고 시원한 음식,"2: 차갑고 새콤한 음식(냉면·비빔국수·쫄면) / 1: 차갑지만 상큼하지 않음, 이름만 상큼 / 0: 뜨거운 음식"
7,포만감 있는 저녁밥,"2: 든든한 밥 식사 / 1: 밥 아닌 식사, 흰밥 / 0: 간식·가벼운 음식"
8,빠르게 먹을 수 있는 간식,2: 간단히 먹는 간식(샌드위치·주먹밥·핫도그·꼬치) / 1: 버거·피자·라면·죽 / 0: 정찬·면 식사
9,따뜻한 국이나 찌개,2: 국·찌개·탕·전골 / 1: 국물이 자작한 요리 / 0: 비국물


In [6]:
# 질의별 적합도 분포
sheet = pd.DataFrame(rows)
sheet["적합도"] = pd.to_numeric(sheet["적합도"], errors="coerce")
dist = sheet.groupby("질의")["적합도"].agg(
    판정수="count", 적합=lambda s: int((s == 2).sum()), 부분=lambda s: int((s == 1).sum()),
    부적합=lambda s: int((s == 0).sum()), 미판정=lambda s: int(s.isna().sum()))
dist.loc[QUERIES]

,판정수,적합,부분,부적합,미판정
질의,,,,,
비 오는 날 얼큰한 국물 먹고 싶어,40,16,20,4,0
맵지 않고 따뜻한 음식,43,33,7,3,0
차갑고 가볍게 먹을 메뉴,51,11,16,24,0
바삭하고 기름진 음식,49,19,10,20,0
든든한 밥 한 끼,43,20,22,1,0
국물 없는 매운 음식,35,18,9,8,0
상큼하고 시원한 음식,40,11,5,24,0
포만감 있는 저녁밥,36,21,15,0,0
빠르게 먹을 수 있는 간식,44,6,19,19,0


## 5. 지표

- P@5: 상위 5개 중 적합(2) 비율
- nDCG@5: 적합도를 이득으로 한 정규화 누적 이득
- MRR: 첫 적합 항목 순위의 역수 평균
- 미판정비율: 반환 항목 중 판정되지 않은 비율. 미판정은 0으로 계산한다.

In [7]:
jmap = judgment_map(rows)
print(f"판정 {len(jmap)}건 (모델 추정 포함)")
CONFIGS = {**MODES, **VARIANTS,
           "선호가중치0": PipelineConfig(ranking=RankingConfig(similarity_weight=1.0, preference_weight=0.0))}
t0 = time.time()
eval_rows = evaluate_configs(RECS, QUERIES, CONFIGS, jmap, k=K)
eval_df = pd.DataFrame(eval_rows).round(4)
print(f"{len(eval_rows)}개 조합 평가, {time.time() - t0:.1f}초")
eval_df.pivot(index="설정", columns="텍스트구성", values=[f"P@{K}", f"nDCG@{K}", "MRR", "미판정비율"]).loc[list(CONFIGS)]

판정 1272건 (모델 추정 포함)


22개 조합 평가, 0.8초


P@5         nDCG@5             MRR          미판정비율      
텍스트구성            A      B       A       B       A       B      A     B
설정                                                                    
임베딩만         0.350  0.425  0.4244  0.5067  0.4883  0.5396  0.000  0.00
조건적용         0.440  0.505  0.5349  0.6189  0.5383  0.6521  0.000  0.00
조건+재랭킹+중복제어  0.695  0.725  0.7729  0.8094  0.8000  0.8771  0.000  0.00
상한0          0.710  0.745  0.7823  0.8223  0.8000  0.8771  0.000  0.00
상한1          0.660  0.710  0.7492  0.7970  0.8000  0.8771  0.025  0.00
감점0.02       0.665  0.710  0.7528  0.7970  0.8000  0.8771  0.020  0.00
후보100 고정     0.655  0.655  0.7336  0.7448  0.7417  0.7688  0.030  0.00
후보400 고정     0.695  0.725  0.7729  0.8094  0.8000  0.8771  0.000  0.00
메뉴가점0        0.645  0.700  0.7339  0.7954  0.7688  0.8562  0.010  0.00
메뉴가점0.3      0.705  0.725  0.7896  0.8234  0.8250  0.9021  0.010  0.00
선호가중치0       0.470  0.485  0.5614  0.5977  0.5654  0.6167  0.015  0.03

## 6. 모드별 관찰

임베딩만 -> 조건적용 -> 전체 파이프라인 순으로 지표가 어떻게 달라지는지 A/B 각각 본다.

In [8]:
eval_df[eval_df["설정"].isin(MODES)].set_index(["텍스트구성", "설정"]).loc[[("A", m) for m in MODES] + [("B", m) for m in MODES]]

질의수    P@5  nDCG@5     MRR  미판정비율
텍스트구성 설정                                            
A     임베딩만          40  0.350  0.4244  0.4883    0.0
      조건적용          40  0.440  0.5349  0.5383    0.0
      조건+재랭킹+중복제어   40  0.695  0.7729  0.8000    0.0
B     임베딩만          40  0.425  0.5067  0.5396    0.0
      조건적용          40  0.505  0.6189  0.6521    0.0
      조건+재랭킹+중복제어   40  0.725  0.8094  0.8771    0.0

## 7. 질의별 상세 (텍스트 B, 전체 파이프라인)

In [9]:
from dataclasses import replace

per_query = []
for q in QUERIES:
    r = RECS["B"].recommend(q, replace(FULL, top_k=K))
    m = evaluate_result(r, jmap, K)
    per_query.append({"유형": QUERY_KIND[q], **m,
                      "상위5": " / ".join(f"{it['메뉴명']}({jmap.get((q, it['라벨링단위ID']), '?')})" for it in r["추천"])})
per_query_df = pd.DataFrame(per_query).round(3)
per_query_df

,유형,질의,반환수,미판정수,P@5,nDCG@5,RR,적합수,부분적합수,상위5
0,5단계 기존,비 오는 날 얼큰한 국물 먹고 싶어,5,0,0.6,0.744,1.000,3,2,수제비 김치(2) / 칼국수(1) / 김치전(1) / 해장국 뼈다귀(2) / 꽃게 매운탕(2)
1,5단계 기존,맵지 않고 따뜻한 음식,5,0,1.0,1.000,1.000,5,0,화양적(2) / 무 된장국(2) / 햄버거(2) / 복지리(2) / 족발(2)
2,5단계 기존,차갑고 가볍게 먹을 메뉴,5,0,0.6,0.790,1.000,3,2,미역냉국 오이 고추(2) / 냉국 미역 오이(2) / 소고기 감자죽(1) / 미소된장국(1) / 국수 김치말이국수(2)
3,5단계 기존,바삭하고 기름진 음식,5,0,1.0,1.000,1.000,5,0,깐풍기(2) / 양념 돼지고기튀김(2) / 돼지고기강정(2) / 참치강정(2) / 비프까스(2)
4,5단계 기존,든든한 밥 한 끼,5,0,1.0,1.000,1.000,5,0,육회비빔밥(2) / 덮밥 닭고기(2) / 소고기 덮밥(2) / 비빔 잡곡밥(2) / 돼지고기 덮밥(2)
5,5단계 기존,국물 없는 매운 음식,5,0,0.6,0.815,1.000,3,2,쟁반국수(2) / 막국수(2) / 국수 쟁반막국수(2) / 황태구이(1) / 해물볶음(1)
6,5단계 기존,상큼하고 시원한 음식,5,0,0.8,0.903,1.000,4,1,국수 김치말이국수(2) / 국수 열무김치(2) / 삼색 오이냉국(2) / 콩국수(1) / 냉면 회냉면 홍어(2)
7,5단계 기존,포만감 있는 저녁밥,5,0,1.0,1.000,1.000,5,0,소고기 덮밥(2) / 덮밥 해물(2) / 하이라이스(2) / 잡탕밥(2) / 자장밥(2)
8,5단계 기존,빠르게 먹을 수 있는 간식,5,0,0.4,0.657,1.000,2,3,주먹밥(2) / 채소죽(1) / 땅콩죽(1) / 채소 꼬치구이(2) / 소고기 감자죽(1)
9,5단계 기존,따뜻한 국이나 찌개,5,0,1.0,1.000,1.000,5,0,두부찌개(2) / 감자 소고기찌개(2) / 섞어찌개(모듬찌개)(2) / 굴 두부찌개(2) / 조기찌개(2)


In [10]:
worst = per_query_df.sort_values(f"nDCG@{K}").head(5)
print("nDCG가 낮은 질의 5개:")
for _, row in worst.iterrows():
    print(f"  {row['질의']!r}: nDCG={row[f'nDCG@{K}']}, P@5={row[f'P@{K}']}, 미판정={row['미판정수']}")
    print(f"     {row['상위5']}")

nDCG가 낮은 질의 5개:
  '샌드위치 가볍게': nDCG=0.0, P@5=0.0, 미판정=0
     땅콩죽(0) / 시금치 된장국(0) / 미소된장국(0) / 오믈렛(0) / 채소죽(0)
  '피자 먹고 싶은데 느끼하지 않은 걸로': nDCG=0.139, P@5=0.0, 미판정=0
     피자 불고기피자(1) / 돼지고기 피망잡채(0) / 회덮밥(0) / 기스면(0) / 애호박죽(0)
  '다이어트 중이라 가벼운 샐러드': nDCG=0.182, P@5=0.0, 미판정=0
     중식잡채(0) / 채소죽(1) / 채소 꼬치구이(1) / 배추국 들깨(0) / 잡채(0)
  '단짠단짠한 음식': nDCG=0.39, P@5=0.4, 미판정=0
     분짜(1) / 짬뽕(0) / 붕어 매운탕(0) / 닭찜(2) / 소고기 떡찜(2)
  '치킨 먹고 싶어': nDCG=0.403, P@5=0.2, 미판정=0
     햄버거 치킨(1) / 치킨가스(1) / 치킨데리야끼(2) / 닭볶음(닭갈비) 매운양념 치즈(1) / 김치 돼지고기볶음(0)


## 8. 설정 비교 요약

차이는 모델 추정 정답 기준이며 승인 후 바뀔 수 있다.

In [11]:
best = eval_df.sort_values(f"nDCG@{K}", ascending=False).head(8)
best[["텍스트구성", "설정", f"P@{K}", f"nDCG@{K}", "MRR", "미판정비율"]]

,텍스트구성,설정,P@5,nDCG@5,MRR,미판정비율
20,B,메뉴가점0.3,0.725,0.8234,0.9021,0.00
14,B,상한0,0.745,0.8223,0.8771,0.00
13,B,조건+재랭킹+중복제어,0.725,0.8094,0.8771,0.00
18,B,후보400 고정,0.725,0.8094,0.8771,0.00
16,B,감점0.02,0.710,0.7970,0.8771,0.00
15,B,상한1,0.710,0.7970,0.8771,0.00
19,B,메뉴가점0,0.700,0.7954,0.8562,0.00
9,A,메뉴가점0.3,0.705,0.7896,0.8250,0.01


In [12]:
# A/B 차이: 같은 설정에서 B - A
ab = eval_df.pivot(index="설정", columns="텍스트구성", values=f"nDCG@{K}")
ab["B-A"] = (ab["B"] - ab["A"]).round(4)
ab.loc[list(CONFIGS)]

텍스트구성,A,B,B-A
설정,,,
임베딩만,0.4244,0.5067,0.0823
조건적용,0.5349,0.6189,0.0840
조건+재랭킹+중복제어,0.7729,0.8094,0.0365
상한0,0.7823,0.8223,0.0400
상한1,0.7492,0.7970,0.0478
감점0.02,0.7528,0.7970,0.0442
후보100 고정,0.7336,0.7448,0.0112
후보400 고정,0.7729,0.8094,0.0365
메뉴가점0,0.7339,0.7954,0.0615


## 9. 설정 차이의 신뢰구간

질의별 nDCG@5 차이를 질의 단위로 복원 추출해 95% 구간을 구한다. 구간이 0을 포함하면 차이를 말할 수 없다.

In [13]:
PQ = {(v, name): evaluate_per_query(RECS[v], QUERIES, cfg, jmap, k=K) for v in RECS for name, cfg in CONFIGS.items()}
nd = lambda v, name: [r[f"nDCG@{K}"] for r in PQ[(v, name)]]
COMPARISONS = [
    ("B 임베딩만", "B 조건적용"), ("B 조건적용", "B 조건+재랭킹+중복제어"), ("B 임베딩만", "B 조건+재랭킹+중복제어"),
    ("A 조건+재랭킹+중복제어", "B 조건+재랭킹+중복제어"), ("B 조건+재랭킹+중복제어", "B 상한0"),
    ("B 조건+재랭킹+중복제어", "B 상한1"), ("B 조건+재랭킹+중복제어", "B 감점0.02"),
    ("B 후보100 고정", "B 조건+재랭킹+중복제어"), ("B 선호가중치0", "B 조건+재랭킹+중복제어"),
    ("B 메뉴가점0", "B 조건+재랭킹+중복제어"), ("B 조건+재랭킹+중복제어", "B 메뉴가점0.3"),
]
ci_rows = []
for a, b in COMPARISONS:
    va, vb = nd(a[0], a[2:]), nd(b[0], b[2:])
    ci_rows.append({"기준": a, "비교": b, **paired_bootstrap(va, vb)})
ci_df = pd.DataFrame(ci_rows).round(4)
ci_df

,기준,비교,평균차이,하한95,상한95,질의수,0포함
0,B 임베딩만,B 조건적용,0.1122,0.0445,0.1947,40,False
1,B 조건적용,B 조건+재랭킹+중복제어,0.1905,0.0764,0.3061,40,False
2,B 임베딩만,B 조건+재랭킹+중복제어,0.3026,0.1735,0.4348,40,False
3,A 조건+재랭킹+중복제어,B 조건+재랭킹+중복제어,0.0364,-0.0030,0.0772,40,True
4,B 조건+재랭킹+중복제어,B 상한0,0.0129,0.0000,0.0316,40,True
5,B 조건+재랭킹+중복제어,B 상한1,-0.0124,-0.0260,-0.0018,40,False
6,B 조건+재랭킹+중복제어,B 감점0.02,-0.0124,-0.0260,-0.0018,40,False
7,B 후보100 고정,B 조건+재랭킹+중복제어,0.0646,0.0086,0.1332,40,False
8,B 선호가중치0,B 조건+재랭킹+중복제어,0.2117,0.1040,0.3220,40,False
9,B 메뉴가점0,B 조건+재랭킹+중복제어,0.0139,-0.0148,0.0522,40,True


In [14]:
# 메뉴 언급이 있는 질의만 따로: 가점의 효과는 여기서만 날 수 있다
mentioned = [q for q in QUERIES if parse_query(q).menu_terms]
print(f"메뉴 언급 질의 {len(mentioned)}개: {mentioned}")
idx = [QUERIES.index(q) for q in mentioned]
sub = lambda v, name: [nd(v, name)[i] for i in idx]
pd.DataFrame([
    {"기준": "B 메뉴가점0", "비교": "B 조건+재랭킹+중복제어 (가점 0.15)", **paired_bootstrap(sub("B", "메뉴가점0"), sub("B", "조건+재랭킹+중복제어"))},
    {"기준": "B 조건+재랭킹+중복제어 (가점 0.15)", "비교": "B 메뉴가점0.3", **paired_bootstrap(sub("B", "조건+재랭킹+중복제어"), sub("B", "메뉴가점0.3"))},
]).round(4)

메뉴 언급 질의 15개: ['든든한 밥 한 끼', '따뜻한 국이나 찌개', '튀김 말고 구운 치킨', '피자 먹고 싶은데 느끼하지 않은 걸로', '따뜻하고 든든한 국밥', '차가운 면 요리', '안 맵고 국물 있는 면', '치킨 먹고 싶어', '피자 말고 버거', '샌드위치 가볍게', '떡볶이 매운 거', '김밥이랑 국물', '추운 날 뜨끈한 국밥', '비 오는 날 전 부쳐 먹고 싶다', '다이어트 중이라 가벼운 샐러드']


,기준,비교,평균차이,하한95,상한95,질의수,0포함
0,B 메뉴가점0,B 조건+재랭킹+중복제어 (가점 0.15),0.0541,-0.0001,0.1547,15,True
1,B 조건+재랭킹+중복제어 (가점 0.15),B 메뉴가점0.3,0.0561,-0.0277,0.1911,15,True


In [15]:
# 유형별 평균 nDCG (B 전체 파이프라인)
type_rows = pd.DataFrame(PQ[("B", "조건+재랭킹+중복제어")])
type_rows["유형"] = [QUERY_KIND[q] for q in type_rows["질의"]]
type_rows.groupby("유형")[[f"P@{K}", f"nDCG@{K}", "RR", "미판정수"]].mean().round(3)

,P@5,nDCG@5,RR,미판정수
유형,,,,
5단계 기존,0.783,0.858,0.938,0.0
5단계 미확정,0.600,0.684,1.000,0.0
5단계 복합,0.667,0.713,0.667,0.0
5단계 부정,0.600,0.780,1.000,0.0
추가 맥락,0.720,0.772,0.700,0.0
추가 메뉴언급,0.440,0.573,0.667,0.0
추가 복합,0.857,0.920,1.000,0.0
추가 부정,0.800,0.905,1.000,0.0


## 10. 파서 처리 결과와 지표

필수 조건이 있는 질의, 선호만 있는 질의, 조건이 없는 질의를 나눠 본다.

In [16]:
def kind_of(q):
    p = parse_query(q)
    if p.hard:
        return "필수 조건 있음"
    if p.soft:
        return "선호만"
    return "조건 없음"

per_query_df["조건유형"] = [kind_of(q) for q in per_query_df["질의"]]
per_query_df.groupby("조건유형")[[f"P@{K}", f"nDCG@{K}", "RR", "미판정수"]].mean().round(3)

,P@5,nDCG@5,RR,미판정수
조건유형,,,,
선호만,0.752,0.828,0.905,0.0
조건 없음,0.480,0.591,0.617,0.0
필수 조건 있음,0.771,0.860,0.929,0.0


## 11. 승인된 판정만으로 계산

검토상태가 승인인 행만 쓴다. 승인이 없으면 실행하지 않는다.

In [17]:
approved = judgment_map(rows, approved_only=True)
approved_queries = sorted({q for q, _ in approved})
if approved:
    print(f"승인 판정 {len(approved)}건, 질의 {len(approved_queries)}개")
    approved_df = pd.DataFrame(evaluate_configs(RECS, approved_queries, CONFIGS, approved, k=K)).round(4)
    display(approved_df.pivot(index="설정", columns="텍스트구성", values=[f"P@{K}", f"nDCG@{K}", "MRR", "미판정비율"]))
else:
    approved_df = None
    print("승인된 판정이 없어 미실행. judgments.csv에서 적합도를 확인하고 검토상태를 '승인'으로 바꾼 뒤 다시 실행한다.")

승인된 판정이 없어 미실행. judgments.csv에서 적합도를 확인하고 검토상태를 '승인'으로 바꾼 뒤 다시 실행한다.


## 12. 결과 저장

In [18]:
eval_df.to_csv(OUT_DIR / "eval_configs.csv", index=False, encoding="utf-8-sig")
per_query_df.to_csv(OUT_DIR / "eval_per_query_B_full.csv", index=False, encoding="utf-8-sig")
ci_df.to_csv(OUT_DIR / "eval_bootstrap.csv", index=False, encoding="utf-8-sig")
run_config = {
    "실행시각": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "k": K, "질의": QUERIES, "제외질의": EXCLUDED,
    "판정": {"파일": str(JUDGMENTS_PATH.relative_to(PROJECT_ROOT)), "행수": len(rows), "검토상태": dict(status),
           "판정출처": dict(Counter(r["판정출처"] for r in rows if r["판정출처"]))},
    "풀설정": {"A,B": list(MODES), "B": list(VARIANTS)},
    "평가설정": {name: asdict(cfg) for name, cfg in CONFIGS.items()},
    "임베딩": {v: rec.embedding_ref for v, rec in RECS.items()},
    "부트스트랩": ci_df.to_dict("records"),
    "주의": "판정은 모델 추정(검토대기) 기준, 미판정은 0으로 계산, 승인 판정 기준 결과는 approved 절 참조",
    "승인기준결과": approved_df.to_dict("records") if approved_df is not None else None,
}
with open(OUT_DIR / "eval_run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2, default=str)
for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name:28s} {p.stat().st_size:>9,} bytes")

eval_bootstrap.csv                 910 bytes
eval_configs.csv                 1,015 bytes
eval_per_query_B_full.csv        7,239 bytes
eval_run_config.json            11,427 bytes
judgments.csv                  382,316 bytes
judgments_adjudicated.csv      197,010 bytes
judgments_crosscheck.csv       212,180 bytes
queries.csv                      1,763 bytes


## 13. 요약과 한계

관찰 (모델 추정 정답 기준)
- 임베딩만, 조건적용, 전체 파이프라인 순으로 nDCG@5가 올라가고 각 단계의 차이는 구간이 0을 벗어난다.
- 선호 재랭킹과 후보 확장도 유의하다. 상한 1과 감점 0.02는 nDCG를 낮춘다.
- B가 A보다 조금 높고 구간이 0을 살짝 벗어나지만, 정답이 모델 추정이라 결론은 승인 후로 미룬다.
- 메뉴 제외는 효과가 분명하다. 메뉴 가점은 평균은 양수지만 구간이 0을 포함해 근거가 약하다.
- 낮은 질의는 대부분 데이터 한계다. 공공 데이터에 피자, 샌드위치, 치킨, 샐러드가 거의 없고, 상큼은 스키마에 없다.

한계
- 정답이 모델 추정이다. 신뢰구간은 질의 표본의 불확실성만 반영한다.
- 풀에 들어간 설정이 유리하다. 새 설정은 풀에 넣고 판정을 추가해야 한다.
- 질의 40개, 유형별 5개 안팎이라 유형별 표는 참고만 한다.

다음
- 판정 시트를 검토하고 승인한 뒤 다시 실행한다.
- 실제 사용자 질의를 모아 정답 세트를 늘린다.